# 1. Preparación, entrenamiento y trazas
Ejecutar en Linux con el kernel de `.venv`. Los datos y modelos quedan locales e ignorados por Git. Este notebook llama a los adaptadores del protocolo; los notebooks históricos son solo referencia.


In [1]:
from pathlib import Path
import platform
import sys
import os
raiz = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'shared' / 'contract.py').is_file()), None)
assert raiz is not None, f'No se encuentra la raíz del repositorio desde {Path.cwd()}'
os.chdir(raiz)
sys.path.insert(0, str(raiz))
from shared.contract import ROOT, validate_splits
assert Path.cwd().resolve() == ROOT
print(platform.platform(), sys.version.split()[0], ROOT)
assert sys.platform.startswith('linux'), 'Entrenar LSTM en el entorno Linux validado'
import tensorflow as tf
import lightgbm
import pyarrow
print('Dependencias:', 'tensorflow', tf.__version__, 'lightgbm', lightgbm.__version__, 'pyarrow', pyarrow.__version__)


Linux-5.15.148-tegra-aarch64-with-glibc2.35 3.10.12 /home/jetson/Desktop/greenhouse-online-anomaly-detection
Dependencias: tensorflow 2.16.1 lightgbm 4.6.0 pyarrow 25.0.1


## Datos físicos congelados
Si faltan los archivos locales, revisar primero su tamaño y procedencia en `data_contract/manifests/imported_files.json`; la hidratación verifica hashes y no modifica la fuente.


In [2]:
# Ejecutar solo si faltan los archivos ignorados y existe el checkout fuente local.
# from shared.data_preparation.hydrate import hydrate
# hydrate(Path('../Clasificaci-n-y-correcci-n'))
contract = validate_splits()
assert [contract['splits'][s]['rows'] for s in ('train', 'validation', 'test')] == [172873, 28800, 31680]
print(contract['data_source'], contract['manifest_id'])


Agroconnect 4cc10ef96d923a326a86c0882a356a4d09d35a6da6553bfdbb60dd80a5f0737c


## LightGBM con noche y contexto interior/exterior
Es un experimento nuevo de 68 variables. El v8 original permanece como referencia de 66 variables.


In [3]:
from approaches.lightgbm.night_context import train as train_lightgbm, export_test as export_lightgbm
lightgbm_run = train_lightgbm()
lightgbm_trace, lightgbm_isolated = export_lightgbm(lightgbm_run)
print(lightgbm_run, lightgbm_trace, lightgbm_isolated, sep='\n')


/home/jetson/Desktop/greenhouse-online-anomaly-detection/experiments/runs/lightgbm_night_v1/20261105T085216Z
/home/jetson/Desktop/greenhouse-online-anomaly-detection/results/traces/lightgbm_night_test.parquet
/home/jetson/Desktop/greenhouse-online-anomaly-detection/results/traces/lightgbm_night_model2_isolated_test.parquet


## LSTM causal
Model 1 y Model 2 se entrenan con train; validation selecciona checkpoints y umbral. Test solo se usa al exportar trazas. Puede tardar bastante.


In [5]:
from approaches.lstm.adapter import train as train_lstm, export_test as export_lstm
lstm_run = train_lstm()
lstm_trace, lstm_isolated = export_lstm(lstm_run)
print(lstm_run, lstm_trace, lstm_isolated, sep='\n')


2026-11-05 10:03:23.299667: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 373364640 exceeds 10% of free system memory.
2026-11-05 10:03:33.521814: W external/local_tsl/tsl/framework/bfc_allocator.cc:487] Allocator (GPU_0_bfc) ran out of memory trying to allocate 356.07MiB (rounded to 373364736)requested by op _EagerConst
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2026-11-05 10:03:33.522201: W external/local_tsl/tsl/framework/bfc_allocator.cc:499] *___________________________________________________________________________________________________


InternalError: Failed copying input tensor from /job:localhost/replica:0/task:0/device:CPU:0 to /job:localhost/replica:0/task:0/device:GPU:0 in order to run _EagerConst: Dst tensor is not initialized.

Guardar las rutas de ambos runs. El notebook 02 verifica la paridad de objetivos y genera el informe comparativo. No versionar las salidas de este notebook.
